# 07 - Feature Engineering Exploration

## Context
Fitur mentah jarang langsung menangkap sinyal prediktif secara optimal. Feature engineering mengubah domain knowledge menjadi input model yang lebih informatif.

## Objective
- Time features dari `TransactionDT`
- Amount features untuk menangkap pola fraud
- Aggregasi level card dan address
- Sinyal dari email domain
- Transaction velocity (deteksi card testing)
- Missingness indicators

### Import & Load Data

In [ ]:
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

TRAIN_PATH = Path('../data/interim/train_merged.parquet')

train = pd.read_parquet(TRAIN_PATH)
print(f'Data loaded: {train.shape}')

### Eksplorasi: Time Features

In [ ]:
# TransactionDT adalah detik yang berlalu sejak tanggal referensi
START_DATE = pd.Timestamp('2017-11-30')
train['TransactionDate'] = START_DATE + pd.to_timedelta(train['TransactionDT'], unit='s')

train['hour']         = train['TransactionDate'].dt.hour
train['day_of_week']  = train['TransactionDate'].dt.dayofweek
train['day_of_month'] = train['TransactionDate'].dt.day
train['is_weekend']   = (train['day_of_week'] >= 5).astype(int)

print(train[['hour', 'day_of_week', 'day_of_month', 'is_weekend']].describe().transpose())

### Visualisasi: Fraud Rate per Jam & Hari

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Fraud rate per jam
hour_fraud = train.groupby('hour')['isFraud'].mean() * 100
axes[0].bar(hour_fraud.index, hour_fraud.values, color='steelblue')
axes[0].axhline(y=train['isFraud'].mean() * 100, color='red', linestyle='--', label='Rata-rata')
axes[0].set_title('Fraud Rate per Jam Transaksi')
axes[0].set_xlabel('Jam')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].legend()

# Fraud rate per hari dalam seminggu
day_labels = ['Sen', 'Sel', 'Rab', 'Kam', 'Jum', 'Sab', 'Min']
dow_fraud = train.groupby('day_of_week')['isFraud'].mean() * 100
axes[1].bar(range(7), dow_fraud.values, color='steelblue')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_labels)
axes[1].axhline(y=train['isFraud'].mean() * 100, color='red', linestyle='--', label='Rata-rata')
axes[1].set_title('Fraud Rate per Hari dalam Seminggu')
axes[1].set_xlabel('Hari')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Fraud rate weekend : {train[train["is_weekend"]==1]["isFraud"].mean()*100:.2f}%')
print(f'Fraud rate weekday : {train[train["is_weekend"]==0]["isFraud"].mean()*100:.2f}%')

### Eksplorasi: Amount Features

In [ ]:
# Transformasi log untuk mengurangi right skew ekstrem
train['TransactionAmt_log']     = np.log1p(train['TransactionAmt'])
# Bagian desimal — jumlah aneh seperti $1.00 bisa tanda card testing
train['TransactionAmt_decimal'] = train['TransactionAmt'] - train['TransactionAmt'].astype(int)
# Jumlah bulat sering dipakai bot untuk verifikasi kartu
train['is_round_amount']        = (train['TransactionAmt'] % 1 == 0).astype(int)

print(train[['TransactionAmt', 'TransactionAmt_log', 'TransactionAmt_decimal', 'is_round_amount']].describe())

### Visualisasi: Distribusi Amount & Fraud Rate Round Amount

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribusi amount mentah
axes[0].hist(train['TransactionAmt'].clip(upper=2000), bins=80, color='#e74c3c', alpha=0.7)
axes[0].set_title('TransactionAmt (Mentah)')
axes[0].set_xlabel('Amount')

# Distribusi setelah log transform
axes[1].hist(train['TransactionAmt_log'], bins=80, color='#2ecc71', alpha=0.7)
axes[1].set_title('TransactionAmt (Log-transform)')
axes[1].set_xlabel('log1p(Amount)')

# Fraud rate: jumlah bulat vs tidak bulat
round_fraud    = train[train['is_round_amount'] == 1]['isFraud'].mean() * 100
nonround_fraud = train[train['is_round_amount'] == 0]['isFraud'].mean() * 100
axes[2].bar(['Bulat', 'Tidak Bulat'], [round_fraud, nonround_fraud], color=['#3498db', '#e67e22'])
axes[2].set_title('Fraud Rate: Jumlah Bulat vs Tidak Bulat')
axes[2].set_ylabel('Fraud Rate (%)')
for i, v in enumerate([round_fraud, nonround_fraud]):
    axes[2].text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

### Eksplorasi: Agregasi Level Card

In [ ]:
# Apakah transaksi ini konsisten dengan perilaku historis kartu ini?
card_agg = (
    train.groupby('card1')['TransactionAmt']
    .agg(['mean', 'std', 'count'])
    .rename(columns={'mean': 'card1_amt_mean', 'std': 'card1_amt_std', 'count': 'card1_txn_count'})
)

# Hapus kolom lama jika sudah ada (aman untuk re-run)
train = train.drop(columns=[c for c in card_agg.columns if c in train.columns], errors='ignore')
train = train.merge(card_agg.reset_index(), on='card1', how='left')

# Selisih dan rasio transaksi saat ini vs rata-rata historis kartu
train['amt_vs_card_mean']      = train['TransactionAmt'] - train['card1_amt_mean']
train['amt_to_card_mean_ratio'] = train['TransactionAmt'] / (train['card1_amt_mean'] + 1)

print(train[['card1_amt_mean', 'card1_amt_std', 'card1_txn_count', 'amt_vs_card_mean', 'amt_to_card_mean_ratio']].describe())

### Visualisasi: Fraud Rate per Jumlah Transaksi Kartu

In [ ]:
# Bin jumlah transaksi per kartu untuk melihat pola fraud
train['card1_count_bin'] = pd.cut(
    train['card1_txn_count'],
    bins=[0, 1, 5, 20, 100, float('inf')],
    labels=['1', '2-5', '6-20', '21-100', '>100']
)
count_fraud = train.groupby('card1_count_bin', observed=True)['isFraud'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(count_fraud.index.astype(str), count_fraud.values, color='steelblue')
ax.axhline(y=train['isFraud'].mean() * 100, color='red', linestyle='--', label='Rata-rata')
ax.set_title('Fraud Rate per Jumlah Transaksi Kartu (card1)')
ax.set_xlabel('Jumlah Transaksi per Kartu')
ax.set_ylabel('Fraud Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

### Eksplorasi: Agregasi Level Alamat

In [ ]:
# Pola transaksi berdasarkan wilayah billing (addr1)
addr_agg = (
    train.groupby('addr1')['TransactionAmt']
    .agg(['mean', 'std', 'count'])
    .rename(columns={'mean': 'addr1_amt_mean', 'std': 'addr1_amt_std', 'count': 'addr1_txn_count'})
)

train = train.drop(columns=[c for c in addr_agg.columns if c in train.columns], errors='ignore')
train = train.merge(addr_agg.reset_index(), on='addr1', how='left')

train['amt_vs_addr_mean'] = train['TransactionAmt'] - train['addr1_amt_mean']

print(train[['addr1_amt_mean', 'addr1_amt_std', 'addr1_txn_count', 'amt_vs_addr_mean']].describe())

### Eksplorasi: Email Domain Features

In [ ]:
FREE_EMAILS = {'gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'aol.com', 'icloud.com'}

P_email = train['P_emaildomain'].astype(str)
R_email = train['R_emaildomain'].astype(str)

# Apakah domain email pembeli dan penerima sama?
train['email_match']     = (P_email == R_email).astype(int)
# Email gratis lebih mudah dibuat massal oleh fraudster
train['P_email_is_free'] = P_email.isin(FREE_EMAILS).astype(int)
train['R_email_is_free'] = R_email.isin(FREE_EMAILS).astype(int)

print(f"email_match rate    : {train['email_match'].mean()*100:.1f}%")
print(f"P_email_is_free rate: {train['P_email_is_free'].mean()*100:.1f}%")
print(f"R_email_is_free rate: {train['R_email_is_free'].mean()*100:.1f}%")

### Visualisasi: Fraud Rate per Email Flag

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat in zip(axes, ['email_match', 'P_email_is_free', 'R_email_is_free']):
    fraud_by_flag = train.groupby(feat)['isFraud'].mean() * 100
    ax.bar([str(int(k)) for k in fraud_by_flag.index], fraud_by_flag.values, color=['#2ecc71', '#e74c3c'])
    ax.set_title(f'Fraud Rate: {feat}')
    ax.set_xlabel(feat)
    ax.set_ylabel('Fraud Rate (%)')
    for i, v in enumerate(fraud_by_flag.values):
        ax.text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

### Eksplorasi: Transaction Velocity

In [ ]:
# Waktu antar transaksi pada kartu yang sama — card testing biasanya dalam hitungan detik
train_sorted = train.sort_values(['card1', 'TransactionDT'])

train_sorted['time_since_last_txn'] = (
    train_sorted.groupby('card1')['TransactionDT']
    .diff()
    .fillna(-1)
)

train = train.merge(
    train_sorted[['TransactionID', 'time_since_last_txn']],
    on='TransactionID',
    how='left'
)

# Transaksi kurang dari 60 detik setelah transaksi sebelumnya di kartu yang sama
train['is_rapid_txn'] = (
    (train['time_since_last_txn'] > 0) &
    (train['time_since_last_txn'] < 60)
).astype(int)

print(f"Rapid transactions (<60 detik): {train['is_rapid_txn'].sum():,} ({train['is_rapid_txn'].mean()*100:.2f}%)")
print(f"Fraud rate rapid txn : {train[train['is_rapid_txn']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate normal txn: {train[train['is_rapid_txn']==0]['isFraud'].mean()*100:.2f}%")

### Eksplorasi: Missingness Indicators

In [ ]:
# Kolom yang missingness-nya berkorelasi dengan fraud (dari analisis notebook 04)
HIGH_SIGNAL_MISSING_COLS = ['id_01', 'id_02', 'id_03', 'id_04', 'id_05',
                             'D4', 'D6', 'D7', 'D8']

for col in HIGH_SIGNAL_MISSING_COLS:
    if col in train.columns:
        train[f'{col}_missing'] = train[col].isnull().astype(int)

missing_indicators = [f'{c}_missing' for c in HIGH_SIGNAL_MISSING_COLS if c in train.columns]
print(f'Missingness indicator dibuat: {len(missing_indicators)}')
print(train[missing_indicators].mean().round(3).to_string())

### Simpan Dataset dengan Fitur Baru

In [ ]:
# Hapus kolom bantu sementara sebelum disimpan
train = train.drop(columns=['TransactionDate', 'card1_count_bin'], errors='ignore')

output_path = Path('../data/processed/train_features.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)
train.to_parquet(output_path, index=False)

new_features = [
    'hour', 'day_of_week', 'day_of_month', 'is_weekend',
    'TransactionAmt_log', 'TransactionAmt_decimal', 'is_round_amount',
    'card1_amt_mean', 'card1_amt_std', 'card1_txn_count', 'amt_vs_card_mean', 'amt_to_card_mean_ratio',
    'addr1_amt_mean', 'addr1_amt_std', 'addr1_txn_count', 'amt_vs_addr_mean',
    'email_match', 'P_email_is_free', 'R_email_is_free',
    'time_since_last_txn', 'is_rapid_txn',
] + missing_indicators

print(f'Saved: {output_path}')
print(f'Shape akhir: {train.shape}')
print(f'Fitur baru: {len(new_features)}')

## Summary

| Kategori | Fitur |
|---|---|
| Time | `hour`, `day_of_week`, `day_of_month`, `is_weekend` |
| Amount | `TransactionAmt_log`, `TransactionAmt_decimal`, `is_round_amount` |
| Card aggregation | `card1_amt_mean/std/count`, `amt_vs_card_mean`, `amt_to_card_mean_ratio` |
| Address aggregation | `addr1_amt_mean/std/count`, `amt_vs_addr_mean` |
| Email | `email_match`, `P_email_is_free`, `R_email_is_free` |
| Velocity | `time_since_last_txn`, `is_rapid_txn` |
| Missingness | `{col}_missing` untuk kolom high-signal |

Dataset tersimpan di `data/processed/train_features.parquet`.